# E3 - LSTM / GRU on image sequences

Rows, columns and 4x4 patches; manual LSTM/GRU cells plus PyTorch reference; compare against E1 baseline.

> **Default:** notebooks run in `SMOKE=True` mode so a fresh clone can validate structure without downloading large datasets. Switch to `SMOKE=False` for real experiments.

## Session 0 - Environment

In [ ]:
from pathlib import Path
import os, sys, torch
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SMOKE = True
SEED = 42
print("repo:", ROOT)
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

## Session 1 - Sequence representations

In [ ]:
import torch
from E3.src.sequence import image_to_sequence
x=torch.randn(2,1,28,28)
for mode in ["rows","columns","patch4"]: print(mode,image_to_sequence(x,mode).shape)

## Session 2 - Manual recurrent cells

In [ ]:
from E3.src.models import build_e3_model
for kind in ["manual_lstm","manual_gru","torch_lstm","torch_gru"]:
    print(kind, build_e3_model(kind,"rows")(x).shape)

## Session 3 - Smoke training

In [ ]:
from torch.utils.data import DataLoader
from common.synthetic import classification_dataset
from common.seed import resolve_device
from common.engine import fit_classifier
ds=classification_dataset(24); dl=DataLoader(ds,batch_size=8,shuffle=True); device=resolve_device(); model=build_e3_model("manual_gru","rows").to(device)
crit=torch.nn.CrossEntropyLoss(); opt=torch.optim.Adam(model.parameters(),lr=1e-3); fit_classifier(model,dl,dl,opt,crit,device,1)[-1]

## Session 4 - Full comparison
Run LSTM and GRU across row/column/patch representations, then compare accuracy, parameter count and runtime with E1 MLP/CNN.